# Government Paperwork Assistant and Contract Explainer


## 1. Check the Kaggle Environment


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mariammaged0/tasheel-assetss/tasheel_icon.jpg
/kaggle/input/datasets/mariammaged0/tasheel-assetss/tasheel_hero.png
/kaggle/input/datasets/mariammaged0/government-paperwork/CLEANING_REPORT.md
/kaggle/input/datasets/mariammaged0/government-paperwork/CLEANING_REPORT_V2.md
/kaggle/input/datasets/mariammaged0/government-paperwork/government_assistant_completed_from_user_data/README.md
/kaggle/input/datasets/mariammaged0/government-paperwork/government_assistant_completed_from_user_data/prompts/structured_json_prompt.txt
/kaggle/input/datasets/mariammaged0/government-paperwork/government_assistant_completed_from_user_data/prompts/rag_answer_prompt.txt
/kaggle/input/datasets/mariammaged0/government-paperwork/government_assistant_completed_from_user_data/prompts/intent_classification_prompt.txt
/kaggle/input/datasets/mariammaged0/government-paperwork/government_assistant_completed_from_user_data/csv/service_id_map.csv
/kaggle/input/datasets/mariammaged0/government-paperw

## 2. Install the Required Libraries

This cell installs LangChain, LangGraph, the quantized-model libraries, Streamlit, and ngrok.


In [4]:
!pip install transformers==4.52.4 langchain langchain-classic langchain-community langchain-core langchain-huggingface faiss-cpu sentence-transformers pypdf pydantic accelerate gptqmodel optimum streamlit pyngrok langgraph


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.0 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of gptqmodel to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.7 MB/s eta 0:00:00:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.3 MB/s eta 0:00:00
  Installing build dependencies ... do

## 3. Hugging Face Login

Run this cell once before creating the application.


In [7]:
import os
from kaggle_secrets import UserSecretsClient

# Read the Hugging Face token from Kaggle Secrets
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. Add it to Kaggle Secrets "
        "and enable access for this notebook."
    )

# Make the token available to Streamlit and other subprocesses
os.environ["HF_TOKEN"] = hf_token

# Import after setting the environment variable
from huggingface_hub import login

# Authenticate without displaying an interactive login window
login(
    token=hf_token,
    add_to_git_credential=False
)

print("Hugging Face authentication completed.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face authentication completed.


## 4. Stop an Old Streamlit Process

This prevents an old application process from continuing to use the same port.


In [5]:
import subprocess
subprocess.run(["pkill", "-f", "streamlit run"], check=False)


CompletedProcess(args=['pkill', '-f', 'streamlit run'], returncode=1)

## 5. Create the Streamlit Application

In [8]:
%%writefile /kaggle/working/app.py

import base64
import hashlib
import json
import mimetypes
import os
import re
from collections import defaultdict
from typing import Any, List, TypedDict

import streamlit as st
import torch
from pydantic import BaseModel, Field
from transformers import AutoTokenizer
from gptqmodel import GPTQModel, BACKEND

from langchain_core.language_models.llms import LLM
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.chains import LLMChain
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END


def render_html_card(html: str):
    """Render HTML cards correctly in Streamlit."""
    cleaned = "\n".join(line.strip() for line in html.strip().splitlines())
    st.markdown(cleaned, unsafe_allow_html=True)


def render_hero_banner(image_path: str):
    """Display the hero image as a full-width banner."""
    mime_type = mimetypes.guess_type(image_path)[0] or "image/png"
    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    st.markdown(
        f'<div class="hero-banner"><img src="data:{mime_type};base64,{encoded}" alt="" /></div>',
        unsafe_allow_html=True,
    )


# ================================================================
# Config
# ================================================================

# Pre-quantized 4-bit GPTQ model
MODEL_NAME = "ModelCloud/Mistral-Nemo-Instruct-2407-gptq-4bit"

BASE_DIR = "/kaggle/input/datasets/mariammaged0/government-paperwork/government_assistant_completed_from_user_data"
KB_DIR = f"{BASE_DIR}/knowledge_base/services"
INTENT_DATASET_PATH = f"{BASE_DIR}/training/intent_dataset.jsonl"

PROMPTS_DIR = f"{BASE_DIR}/prompts"

ASSETS_DIR = "/kaggle/input/datasets/mariammaged0/tasheel-assetss"
ICON_PATH = f"{ASSETS_DIR}/tasheel_icon.jpg"
HERO_PATH = next(
    (p for p in [
        f"{ASSETS_DIR}/tasheel_hero.png",
        f"{ASSETS_DIR}/tasheel_hero.jpg",
        f"{ASSETS_DIR}/tasheel_hero.jpeg",
    ] if os.path.exists(p)),
    f"{ASSETS_DIR}/tasheel_hero.png",
)

# ---------------------------------------------------------
# Bilingual interface text
# ---------------------------------------------------------
LABELS = {
    "ar": {
        "lang_name": "English",
        "brand": "دليلك للعقود و الأوراق الرسمية",
        "mode_service": "الخدمات الحكومية",
        "mode_contract": "شرح العقود",
        "service_heading": "إيه الورقة اللي محتاج تستخرجها؟",
        "input_placeholder": "مثال: عايز أجدد البطاقة",
        "ask_button": "استفسر",
        "searching": "جاري البحث عن الخدمة...",
        "unclear": "لم أستطع تحديد الخدمة بدقة، برجاء توضيح الطلب أكثر (مثال: عايز أجدد البطاقة).",
        "contract_mode_hint": 'يبدو إنك عايز تفهم عقد، اختاري mode "شرح العقود" فوق وارفعي ملف الـ PDF.',
        "parse_failed": "تعذّر تنسيق الإجابة، برجاء إعادة المحاولة.",
        "documents_label": "المستندات المطلوبة",
        "fees_label": "الرسوم",
        "fees_average_label": "متوسط الرسوم التقريبي",
        "fees_disclaimer_label": "تنبيه بخصوص الرسوم",
        "steps_label": "الخطوات",
        "where_label": "أقرب مكان للتقديم",
        "official_link_label": "الرابط الرسمي",
        "booking_link_label": "رابط الحجز / التقديم",
        "contract_heading": "ارفع العقد وهنشرحلك أهم النقاط قبل التوقيع",
        "upload_label": "ارفع العقد",
        "analyze_button": "حلل العقد",
        "analyzing": "جاري تحليل العقد...",
        "contract_error": "تعذّر تحليل العقد، برجاء المحاولة مرة أخرى.",
        "sources_label": "المصادر والنصوص المسترجعة",
        "source_label": "المصدر",
        "page_label": "صفحة",
        "citation_warning": "تم عرض المصادر، لكن الموديل لم يلتزم بصيغة الاستشهاد المطلوبة في كل الإجابة.",
        "debug_label": "تفاصيل تقنية (للمساعدة في تشخيص المشكلة)",
        "model_language": "Arabic",
        "dir": "rtl",
        "align": "right",
    },
    "en": {
        "lang_name": "العربية",
        "brand": "Paperwork Assistant",
        "mode_service": "Government Services",
        "mode_contract": "Contract Explainer",
        "service_heading": "Which document do you need?",
        "input_placeholder": "e.g. I want to renew my national ID",
        "ask_button": "Ask",
        "searching": "Looking up the service...",
        "unclear": "I couldn't identify the service clearly. Please rephrase, e.g. 'renew my national ID'.",
        "contract_mode_hint": 'It looks like you\'re asking about a contract - switch to "Contract Explainer" above and upload the PDF.',
        "parse_failed": "Could not format the answer. Please try again.",
        "documents_label": "Required documents",
        "fees_label": "Fees",
        "fees_average_label": "Approximate average fee",
        "fees_disclaimer_label": "Note on fees",
        "steps_label": "Steps",
        "where_label": "Nearest place to apply",
        "official_link_label": "Official link",
        "booking_link_label": "Booking / application link",
        "contract_heading": "Upload a contract and we'll explain the key points before you sign",
        "upload_label": "Upload contract",
        "analyze_button": "Analyze contract",
        "analyzing": "Analyzing the contract...",
        "contract_error": "Could not analyze the contract. Please try again.",
        "sources_label": "Sources and retrieved excerpts",
        "source_label": "Source",
        "page_label": "Page",
        "citation_warning": "The sources are shown below, but the model did not use the required citation format throughout the answer.",
        "debug_label": "Technical details (for debugging)",
        "model_language": "English",
        "dir": "ltr",
        "align": "left",
    },
}


st.set_page_config(
    page_title="Paperwork Assistant / دليلك للعقود و الأوراق الرسمية",
    page_icon=ICON_PATH if os.path.exists(ICON_PATH) else "📄",
    layout="wide",
    initial_sidebar_state="collapsed",
)

if "lang" not in st.session_state:
    st.session_state["lang"] = "ar"

t = LABELS[st.session_state["lang"]]

# ---------------------------------------------------------
# Page style
# ---------------------------------------------------------
st.markdown(
    f"""
    <style>
    :root {{
        --accent: #7a1f2b;
        --accent-dark: #591620;
        --accent-soft: #f7e9e8;
        --background: #f5f7f8;
        --surface: #ffffff;
        --border: #dfe7e3;
        --text: #17251f;
        --muted: #5b6b64;
        --hero-banner-height: 220px;
    }}

    [data-testid="stHeader"] {{ background: transparent; height: 0; }}
    [data-testid="stToolbar"] {{ display: none; }}
    #MainMenu {{ visibility: hidden; }}
    footer {{ visibility: hidden; }}

    .stApp {{
        background: var(--background);
        color: var(--text);
    }}

    .block-container {{
        max-width: 1180px;
        padding-top: 1rem;
        padding-bottom: 3rem;
    }}

    body, .stApp {{ direction: {t['dir']}; }}

    h1, h2, h3, h4 {{ color: var(--text); text-align: {t['align']}; }}

    p, label, span {{ color: var(--muted); }}

    .brand {{ font-size: 1.3rem; font-weight: 800; color: var(--accent-dark); }}

    div[role="radiogroup"] {{ display: flex; gap: 0.4rem; flex-wrap: wrap; }}

    div[role="radiogroup"] label {{
        padding: 0.45rem 1rem;
        border-radius: 999px;
        border: 1px solid var(--border);
        background: var(--surface);
        color: var(--muted);
        font-weight: 700;
    }}

    div[role="radiogroup"] label:has(input:checked) {{
        background: var(--accent-soft);
        border-color: #e3c0c2;
        color: var(--accent-dark);
    }}

    .card {{
        padding: 1.3rem 1.5rem;
        margin-top: 1rem;
        background: var(--surface);
        border: 1px solid var(--border);
        border-radius: 18px;
        box-shadow: 0 6px 20px rgba(15, 23, 42, 0.04);
        text-align: {t['align']};
    }}

    .service-title {{ margin-bottom: 0.6rem; color: var(--accent-dark); font-size: 1.6rem; font-weight: 800; }}

    .field-label {{
        margin-top: 0.9rem; color: var(--muted); font-size: 0.8rem;
        font-weight: 700; letter-spacing: 0.03em; text-transform: uppercase;
    }}

    .field-value {{ margin-top: 0.25rem; color: var(--text); font-size: 1rem; line-height: 1.8; }}

    .chip {{
        display: inline-block; margin: 0.2rem 0.35rem 0.2rem 0; padding: 0.4rem 0.8rem;
        color: var(--accent-dark); background: var(--accent-soft); border: 1px solid #e3c0c2;
        border-radius: 999px; font-size: 0.88rem; font-weight: 600;
    }}

    .step-line {{ margin: 0.35rem 0; color: var(--text); line-height: 1.7; }}

    .stButton > button {{
        min-height: 46px; border: none; border-radius: 11px;
        background: var(--accent); color: white !important; font-weight: 700; width: 100%;
    }}

    /* Keep button text white. */
    .stButton > button p,
    .stButton > button span,
    .stButton > button div {{ color: white !important; }}

    .stButton > button:hover:not(:disabled) {{ background: var(--accent-dark); color: white !important; }}

    .stButton > button:disabled {{ background: #d8b3b6 !important; color: white !important; opacity: 1; }}

    [data-testid="stFileUploader"] {{
        padding: 0.5rem; background: var(--surface); border: 1px dashed #c98f95; border-radius: 16px;
    }}

    [data-testid="stTextInput"] input {{ text-align: {t['align']}; }}

    /* Full-width hero banner. */
    .hero-banner {{
        width: 100%;
        height: var(--hero-banner-height);
        border-radius: 18px;
        overflow: hidden;
        margin-bottom: 1rem;
    }}

    .hero-banner img {{
        width: 100%;
        height: 100%;
        object-fit: cover;
        object-position: center;
        display: block;
    }}

    @media (max-width: 900px) {{
        .hero-banner {{ height: 140px; }}
    }}
    </style>
    """,
    unsafe_allow_html=True,
)


# ---------------------------------------------------------
# Model
# ---------------------------------------------------------

@st.cache_resource(show_spinner="Loading model... / جاري تحميل الموديل...")
def load_model():
    token = os.getenv("HF_TOKEN") or None
    device = "cuda:0" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = GPTQModel.load(
        MODEL_NAME,
        device=device,
        trust_remote_code=True,
        backend=BACKEND.TRITON,
    )
    if hasattr(model, "eval"):
        model.eval()
    return tokenizer, model


tokenizer, model = load_model()


def generate_text(prompt, max_new_tokens=300, do_sample=False):
    """Generate the model response without repeating the prompt."""
    messages = [{"role": "user", "content": prompt}]
    chat_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(chat_prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        gen_kwargs.update(top_k=50, top_p=0.95, temperature=0.4)

    with torch.inference_mode():
        output = model.generate(**inputs, **gen_kwargs)

    generated_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop: Any = None) -> str:
        return generate_text(prompt, max_new_tokens=20)

    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"


llm = CustomHFLLM()


# ---------------------------------------------------------
# RAG backend
# ---------------------------------------------------------

@st.cache_resource(show_spinner=False)
def load_embedding_model():
    """Load one embedding model for both RAG pipelines."""
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )


@st.cache_resource(show_spinner="Preparing knowledge base... / جاري تجهيز قاعدة المعرفة...")
def load_vectordb():
    loader = DirectoryLoader(
        KB_DIR,
        glob="*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    documents = loader.load()
    text_splitter = CharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
        separator="\n",
    )
    chunks = text_splitter.split_documents(documents)
    return FAISS.from_documents(chunks, load_embedding_model())


vectordb = load_vectordb()


@st.cache_resource(show_spinner=False)
def load_intent_examples():
    with open(INTENT_DATASET_PATH, encoding="utf-8") as f:
        intent_records = [json.loads(line) for line in f]

    examples_by_service = defaultdict(list)
    for r in intent_records:
        examples_by_service[r["output"]].append({"user_input": r["input"], "service_id": r["output"]})

    few_shot = []
    for service_id, examples in examples_by_service.items():
        few_shot.extend(examples[:2])

    return few_shot, list(examples_by_service.keys())


few_shot_examples, allowed_service_ids = load_intent_examples()

example_prompt = PromptTemplate(
    input_variables=["user_input", "service_id"],
    template="User request: {user_input}\nService ID: {service_id}",
)

intent_prompt = FewShotPromptTemplate(
    examples=few_shot_examples,
    example_prompt=example_prompt,
    prefix="""Classify the user's request into one service_id.
Reply with ONLY the exact service_id string from the allowed list below, nothing else.

Allowed service IDs:
{service_ids}

Examples:""",
    suffix="""User request: {user_input}
Service ID:""",
    input_variables=["user_input", "service_ids"],
)

intent_chain = LLMChain(llm=llm, prompt=intent_prompt)


# ---------------------------------------------------------
# Deterministic pre-classification (Arabic spelling normalization)
# ---------------------------------------------------------

_ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")


def normalize_arabic(text: str) -> str:
    """Normalize common Arabic letter variations."""
    if not text:
        return ""
    text = _ARABIC_DIACRITICS.sub("", text)
    text = text.replace("\u0640", "")  # Remove tatweel
    text = re.sub(r"[إأآا]", "ا", text)  # Normalize alef
    text = text.replace("ى", "ي")  # Normalize yaa
    text = text.replace("ة", "ه")  # Normalize taa marbuta
    text = re.sub(r"\s+", " ", text).strip()
    return text


@st.cache_resource(show_spinner=False)
def load_intent_lookup():
    with open(INTENT_DATASET_PATH, encoding="utf-8") as f:
        records = [json.loads(line) for line in f]

    exact_lookup = {}
    keyworded_examples = []
    for r in records:
        norm = normalize_arabic(r["input"])
        exact_lookup[norm] = r["output"]
        keyworded_examples.append((set(norm.split()), r["output"]))

    return exact_lookup, keyworded_examples


intent_exact_lookup, intent_keyword_examples = load_intent_lookup()


def keyword_match_intent(user_input, min_overlap=0.6):
    """Match the request with known intent examples."""
    norm = normalize_arabic(user_input)
    if norm in intent_exact_lookup:
        return intent_exact_lookup[norm]

    user_words = set(norm.split())
    if not user_words:
        return None

    best_service, best_score = None, 0.0
    for example_words, service_id in intent_keyword_examples:
        if not example_words:
            continue
        overlap = len(user_words & example_words) / len(user_words | example_words)
        if overlap > best_score:
            best_score, best_service = overlap, service_id

    return best_service if best_score >= min_overlap else None


def classify_intent(user_input):
    matched = keyword_match_intent(user_input)
    if matched:
        return matched, f"[matched by keyword normalization -> {matched}]"

    raw = intent_chain.run(
        user_input=user_input,
        service_ids="\n".join("- " + s for s in allowed_service_ids),
    )
    for service_id in allowed_service_ids:
        if service_id in raw:
            return service_id, raw
    return "general_government_help", raw


class GovernmentServiceAnswer(BaseModel):


    service_name: str = Field(description="Name of the government service, in the requested language")
    required_documents: List[str] = Field(description="List of required documents")
    fees: str = Field(description="Breakdown of the fees by type/speed/category, taken from the context")
    fees_average: str = Field(description="The average or typical fee estimate for this service, taken from the context")
    fees_disclaimer: str = Field(
        description="A note that the fees are 2026 demo data, approximate, and subject to change"
    )
    steps: List[str] = Field(description="The steps needed to apply for the service")
    where_to_apply: str = Field(description="Where the user can apply / nearest place")
    official_link: str = Field(description="The official government link")
    booking_link: str = Field(description="The link to book / apply online")
    warning: str = Field(description="A warning to verify the official link before paying or visiting")


output_parser = PydanticOutputParser(pydantic_object=GovernmentServiceAnswer)
format_instructions = output_parser.get_format_instructions()


def load_fee_disclosure_rules():
    """Load the external rules used by the government RAG prompt."""
    path = f"{PROMPTS_DIR}/rag_answer_prompt.txt"
    try:
        with open(path, encoding="utf-8") as f:
            return f.read().strip()
    except FileNotFoundError:
        # Use default rules if the prompt file is missing.
        return (
            "اعرض الرسوم كبيانات ديمو لسنة 2026 تقديرية وقابلة للتغيير.\n"
            "اعرض متوسط/مدى الرسوم إذا كان موجوداً في السياق.\n"
            "اعرض دائماً الرابط الرسمي ورابط الحجز/التقديم إن كان موجوداً.\n"
            "لا تذكر للمستخدم عبارات مثل: كما ورد في الملف، حسب ملف المستخدم، الملف المرفوع.\n"
            "اختم بتنبيه: تأكد دائماً من الرابط الرسمي قبل الذهاب أو الدفع."
        )


FEE_DISCLOSURE_RULES = load_fee_disclosure_rules()

rag_answer_template = """You are a government paperwork assistant for Egyptian citizens.
Use ONLY the retrieved context below to answer.
Respond entirely in {language}.
If information is missing, say it is not clearly available in the knowledge base.
Return ONLY the JSON object, nothing else before or after it.

Fee-disclosure rules (always follow these, regardless of the response language):
""" + FEE_DISCLOSURE_RULES + """

In particular:
- "fees" must break down the fee by type/speed/category exactly as found in the context (e.g. normal vs urgent vs online).
- "fees_average" must state the average/typical estimate found in the context (look for a range or average figure).
- "fees_disclaimer" must say the fees are approximate 2026 demo data and may change.
- "booking_link" must always be filled in from the context, even if it is the same URL as "official_link".
- "warning" must remind the user to confirm the official link before paying or visiting.

Context:
{context}

Question:
{question}

{format_instructions}
"""

rag_prompt = PromptTemplate(
    template=rag_answer_template,
    input_variables=["context", "question", "language"],
    partial_variables={"format_instructions": format_instructions},
)


def extract_json_block(text):
    pattern = r"```json\s*(.*?)\s*```"
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return matches[-1]
    start, end = text.find("{"), text.rfind("}")
    return text[start:end + 1]


def load_full_service_context(service_id, user_input, k=3):
    """Load the selected service file or use FAISS as fallback."""
    service_path = f"{KB_DIR}/{service_id}.txt"
    if os.path.exists(service_path):
        with open(service_path, encoding="utf-8") as f:
            return f.read()

    # Use vector search if no matching service file exists.
    docs = vectordb.similarity_search(user_input, k=k)
    service_docs = [d for d in docs if d.metadata.get("source", "").find(service_id) != -1]
    if not service_docs:
        service_docs = vectordb.similarity_search(service_id, k=k)
    return "\n\n".join(d.page_content for d in service_docs)


def answer_service_question(service_id, user_input, language, k=3):
    context = load_full_service_context(service_id, user_input, k=k)

    prompt = rag_prompt.format(context=context, question=user_input, language=language)
    response = generate_text(prompt, max_new_tokens=800)

    json_text = extract_json_block(response)
    try:
        return output_parser.parse(json_text), response
    except Exception:
        return None, response


def government_assistant_backend(user_input, language):
    service_id, intent_raw = classify_intent(user_input)

    if service_id == "general_government_help":
        return None, "unclear", service_id, intent_raw

    if service_id == "contract_explainer":
        return None, "contract_mode", service_id, intent_raw

    result, raw_response = answer_service_question(service_id, user_input, language)
    if result is None:
        return None, "parse_failed", service_id, raw_response

    return result, "ok", service_id, None


# ---------------------------------------------------------
# Contract RAG and citations
# ---------------------------------------------------------

def extract_contract_documents(pdf_path):
    """Read all text pages from the uploaded PDF."""
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    usable_documents = [
        document
        for document in documents
        if document.page_content and document.page_content.strip()
    ]

    if not usable_documents:
        raise ValueError(
            "No extractable text was found in the PDF. "
            "The file may be scanned and would require OCR."
        )

    return usable_documents


CONTRACT_RETRIEVAL_QUERIES = [
    "أطراف العقد والغرض ونطاق العمل والخدمات parties purpose scope of work services",
    "المبالغ والدفع والرسوم والضرائب financial obligations payment fees price taxes",
    "مدة العقد والتجديد والإنهاء والإشعار duration renewal termination notice",
    "الغرامات والجزاءات والتعويض والمسؤولية penalties damages indemnity liability",
    "التزامات الأطراف والضمانات والسرية obligations warranties confidentiality",
    "حل النزاعات والقانون والمحكمة والتحكيم disputes governing law jurisdiction arbitration",
]


def build_contract_context(documents, max_chunks=8):
    """Retrieve the most relevant clauses and keep their page numbers."""
    splitter = CharacterTextSplitter(
        chunk_size=1200,
        chunk_overlap=150,
        separator="\n",
    )

    chunks = splitter.split_documents(documents)
    if not chunks:
        raise ValueError("The PDF did not produce any usable text chunks.")

    contract_vectordb = FAISS.from_documents(
        chunks,
        load_embedding_model(),
    )

    # Keep the best score for every unique chunk.
    candidates = {}
    per_query_k = min(3, len(chunks))

    for query in CONTRACT_RETRIEVAL_QUERIES:
        results = contract_vectordb.similarity_search_with_score(
            query,
            k=per_query_k,
        )

        for document, score in results:
            content_key = re.sub(
                r"\s+",
                " ",
                document.page_content,
            ).strip()

            current = candidates.get(content_key)
            if current is None or score < current[0]:
                candidates[content_key] = (float(score), document)

    # Keep the opening section, then add the strongest retrieved clauses.
    max_chunks = max(1, max_chunks)
    first_chunk = chunks[0]
    first_key = re.sub(
        r"\s+",
        " ",
        first_chunk.page_content,
    ).strip()

    first_candidate = candidates.get(
        first_key,
        (float("inf"), first_chunk),
    )

    ranked_candidates = sorted(
        (
            candidate
            for content_key, candidate in candidates.items()
            if content_key != first_key
        ),
        key=lambda item: item[0],
    )

    selected = [first_candidate]
    selected.extend(ranked_candidates[: max_chunks - 1])

    context_parts = []
    sources = []

    for source_id, (_, document) in enumerate(selected, start=1):
        page = document.metadata.get("page")
        page_number = page + 1 if isinstance(page, int) else "unknown"
        source_label = f"[Source {source_id} | Page {page_number}]"
        source_text = document.page_content.strip()

        context_parts.append(
            f"{source_label}\n{source_text}"
        )

        sources.append({
            "id": source_id,
            "page": page_number,
            "label": source_label,
            "text": source_text,
        })

    return (
        "\n\n".join(context_parts),
        sources,
        len(chunks),
        len(selected),
    )


contract_explainer_template = """You are a legal-document assistant helping a non-lawyer understand a contract
before they sign it. Do NOT give binding legal advice. Respond entirely in {language}.

Use ONLY the retrieved excerpts below. Do not invent missing terms.
If a requested point is not present, say that it was not clearly found.

Every excerpt starts with an exact citation label such as:
[Source 2 | Page 7]

Citation rules:
- Add the exact citation label after every major factual point or risk.
- Copy citation labels exactly as they appear in the excerpts.
- Never create a source number or page number that is not shown below.

Retrieved contract excerpts:
{contract_context}

Return:
1. A short summary of the contract.
2. The most important points to ask about before signing.
3. Risky or unclear clauses, especially payment, penalties, duration,
   renewal, termination, liability, confidentiality, and disputes.
4. A reminder to consult a qualified lawyer before the final signature.

{language_reminder}
"""

# Written natively in the target language (not just naming it in English),
# since the model follows the language of the text immediately before
# generation far more reliably than an English sentence describing which
# language to use. Also explicitly bans stray English filler words
# (numbers, dates, amounts) that kept leaking into otherwise-Arabic answers.
LANGUAGE_REMINDERS = {
    "Arabic": (
        "تنبيه أخير مهم جدًا: اكتب الإجابة كاملة باللغة العربية الفصحى فقط. "
        "ممنوع استخدام أي كلمة إنجليزية إطلاقًا، حتى في الأرقام والتواريخ والمبالغ "
        "(اكتبها بالعربي بالكامل، مثال: من 1 أغسطس 2026 إلى 31 يناير 2027، بمبلغ "
        "125,000 جنيهاً)، حتى لو كانت المقتطفات أعلاه فيها كلمات إنجليزية."
    ),
    "English": (
        "Final reminder: write the entire answer only in English. "
        "Do not use any Arabic words, even if the excerpts above contain some."
    ),
}


def get_language_reminder(language):
    return LANGUAGE_REMINDERS.get(language, LANGUAGE_REMINDERS["English"])


contract_prompt = PromptTemplate(
    template=contract_explainer_template,
    input_variables=["contract_context", "language", "language_reminder"],
)


def validate_contract_citations(answer, sources):
    """Check that the answer uses only citation labels from the retrieved sources."""
    allowed_labels = {
        source["label"]
        for source in sources
    }

    used_labels = set(
        re.findall(
            r"\[Source \d+ \| Page [^\]]+\]",
            answer or "",
        )
    )

    invalid_labels = used_labels - allowed_labels
    citations_valid = bool(used_labels) and not invalid_labels

    return citations_valid, sorted(used_labels)


# ---------------------------------------------------------
# LangGraph workflow for contract analysis
# ---------------------------------------------------------

class ContractState(TypedDict, total=False):
    pdf_path: str
    language: str
    max_chunks: int
    documents: List[Any]
    contract_context: str
    sources: List[dict]
    total_chunks: int
    retrieved_chunks: int
    explanation: str
    citations_valid: bool
    used_citations: List[str]
    error: str


def extract_contract_node(state: ContractState):
    """Load the PDF text."""
    try:
        documents = extract_contract_documents(
            state["pdf_path"]
        )
        return {
            "documents": documents,
            "error": "",
        }
    except Exception as error:
        return {
            "error": str(error),
        }


def retrieve_contract_node(state: ContractState):
    """Build FAISS and retrieve the important contract clauses."""
    try:
        (
            contract_context,
            sources,
            total_chunks,
            retrieved_chunks,
        ) = build_contract_context(
            state["documents"],
            max_chunks=state.get("max_chunks", 8),
        )

        return {
            "contract_context": contract_context,
            "sources": sources,
            "total_chunks": total_chunks,
            "retrieved_chunks": retrieved_chunks,
            "error": "",
        }
    except Exception as error:
        return {
            "error": str(error),
        }


def generate_contract_node(state: ContractState):
    """Generate one contract explanation from the retrieved context."""
    try:
        prompt = contract_prompt.format(
            contract_context=state["contract_context"],
            language=state["language"],
            language_reminder=get_language_reminder(state["language"]),
        )

        explanation = generate_text(
            prompt,
            max_new_tokens=900,
        )

        return {
            "explanation": explanation,
            "error": "",
        }
    except Exception as error:
        return {
            "error": str(error),
        }


def validate_citations_node(state: ContractState):
    """Validate citation labels without another LLM call."""
    citations_valid, used_citations = validate_contract_citations(
        state.get("explanation", ""),
        state.get("sources", []),
    )

    return {
        "citations_valid": citations_valid,
        "used_citations": used_citations,
    }


def route_after_node(state: ContractState):
    """Stop the graph when a previous step returned an error."""
    return "stop" if state.get("error") else "continue"


contract_graph_builder = StateGraph(ContractState)

contract_graph_builder.add_node(
    "extract_pdf",
    extract_contract_node,
)
contract_graph_builder.add_node(
    "retrieve_clauses",
    retrieve_contract_node,
)
contract_graph_builder.add_node(
    "generate_answer",
    generate_contract_node,
)
contract_graph_builder.add_node(
    "validate_citations",
    validate_citations_node,
)

contract_graph_builder.add_edge(
    START,
    "extract_pdf",
)

contract_graph_builder.add_conditional_edges(
    "extract_pdf",
    route_after_node,
    {
        "continue": "retrieve_clauses",
        "stop": END,
    },
)

contract_graph_builder.add_conditional_edges(
    "retrieve_clauses",
    route_after_node,
    {
        "continue": "generate_answer",
        "stop": END,
    },
)

contract_graph_builder.add_conditional_edges(
    "generate_answer",
    route_after_node,
    {
        "continue": "validate_citations",
        "stop": END,
    },
)

contract_graph_builder.add_edge(
    "validate_citations",
    END,
)

contract_graph = contract_graph_builder.compile()


def contract_explainer_backend(
    pdf_path,
    language,
    max_chunks=8,
):
    """Run the complete contract graph."""
    result = contract_graph.invoke({
        "pdf_path": pdf_path,
        "language": language,
        "max_chunks": max_chunks,
        "error": "",
    })

    if result.get("error"):
        raise ValueError(result["error"])

    return (
        result["explanation"],
        result["total_chunks"],
        result["retrieved_chunks"],
        result["sources"],
        result["citations_valid"],
    )


# ---------------------------------------------------------
# Hero banner (shown above the top nav bar)
# ---------------------------------------------------------

if os.path.exists(HERO_PATH):
    render_hero_banner(HERO_PATH)


# Navigation

with st.container(border=True):
    brand_col, mode_col, lang_col = st.columns([1.6, 2.4, 1], vertical_alignment="center")

    with brand_col:
        logo_col, name_col = st.columns([1, 3], gap="small")
        with logo_col:
            if os.path.exists(ICON_PATH):
                st.image(ICON_PATH, width=44)
        with name_col:
            st.markdown(f'<div class="brand">{t["brand"]}</div>', unsafe_allow_html=True)

    with mode_col:
        mode = st.radio(
            "mode",
            [t["mode_service"], t["mode_contract"]],
            horizontal=True,
            label_visibility="collapsed",
        )

    with lang_col:
        if st.button(f"🌐 {t['lang_name']}", use_container_width=True):
            st.session_state["lang"] = "en" if st.session_state["lang"] == "ar" else "ar"
            st.rerun()


# ---------------------------------------------------------
# Mode 1: Government services
# ---------------------------------------------------------

if mode == t["mode_service"]:
    st.markdown(f"### {t['service_heading']}")

    user_input = st.text_input(
        "request",
        placeholder=t["input_placeholder"],
        label_visibility="collapsed",
    )

    ask_clicked = st.button(t["ask_button"], type="primary", disabled=not user_input.strip())

    if ask_clicked:
        with st.spinner(t["searching"]):
            result, status, service_id, raw_response = government_assistant_backend(user_input, t["model_language"])

        st.session_state["service_result"] = result
        st.session_state["service_status"] = status
        st.session_state["service_id"] = service_id
        st.session_state["raw_response"] = raw_response

    if st.session_state.get("service_status"):
        status = st.session_state["service_status"]
        result = st.session_state.get("service_result")

        if status == "unclear":
            st.warning(t["unclear"])
            raw = st.session_state.get("raw_response")
            if raw:
                with st.expander(t["debug_label"]):
                    st.code(raw)

        elif status == "contract_mode":
            st.info(t["contract_mode_hint"])

        elif status == "parse_failed" or result is None:
            st.error(t["parse_failed"])
            raw = st.session_state.get("raw_response")
            if raw:
                with st.expander(t["debug_label"]):
                    st.code(raw)

        else:
            documents_html = "".join(f'<span class="chip">{doc}</span>' for doc in result.required_documents)
            steps_html = "".join(
                f'<div class="step-line">{i}. {step}</div>'
                for i, step in enumerate(result.steps, 1)
            )

            render_html_card(f"""
                <div class="card">
                    <div class="service-title">{result.service_name}</div>
                    <div class="field-label">{t['documents_label']}</div>
                    <div class="field-value">{documents_html}</div>
                    <div class="field-label">{t['fees_label']}</div>
                    <div class="field-value">{result.fees}</div>
                    <div class="field-label">{t['fees_average_label']}</div>
                    <div class="field-value">{result.fees_average}</div>
                    <div class="field-label">{t['fees_disclaimer_label']}</div>
                    <div class="field-value">{result.fees_disclaimer}</div>
                    <div class="field-label">{t['steps_label']}</div>
                    <div class="field-value">{steps_html}</div>
                    <div class="field-label">{t['where_label']}</div>
                    <div class="field-value">{result.where_to_apply}</div>
                    <div class="field-label">{t['official_link_label']}</div>
                    <div class="field-value"><a href="{result.official_link}" target="_blank">{result.official_link}</a></div>
                    <div class="field-label">{t['booking_link_label']}</div>
                    <div class="field-value"><a href="{result.booking_link}" target="_blank">{result.booking_link}</a></div>
                </div>
            """)

            st.warning(result.warning)


# ---------------------------------------------------------
# Mode 2: Contract explainer
# ---------------------------------------------------------

if mode == t["mode_contract"]:
    st.markdown(f"### {t['contract_heading']}")

    uploaded_contract = st.file_uploader(
        t["upload_label"],
        type=["pdf"],
        label_visibility="collapsed",
    )

    analyze_clicked = st.button(
        t["analyze_button"],
        type="primary",
        disabled=uploaded_contract is None,
    )

    current_contract_hash = None
    uploaded_bytes = None

    if uploaded_contract is not None:
        uploaded_bytes = uploaded_contract.getvalue()
        current_contract_hash = hashlib.sha256(
            uploaded_bytes
        ).hexdigest()

    if analyze_clicked and uploaded_contract is not None:
        temp_path = (
            f"/kaggle/working/"
            f"contract_{current_contract_hash[:16]}.pdf"
        )

        with open(temp_path, "wb") as file:
            file.write(uploaded_bytes)

        with st.spinner(t["analyzing"]):
            try:
                (
                    explanation,
                    total_chunks,
                    retrieved_chunks,
                    sources,
                    citations_valid,
                ) = contract_explainer_backend(
                    temp_path,
                    t["model_language"],
                    max_chunks=8,
                )

                st.session_state["contract_explanation"] = explanation
                st.session_state["contract_file_hash"] = current_contract_hash
                st.session_state["contract_sources"] = sources
                st.session_state["contract_citations_valid"] = citations_valid
                st.session_state["contract_rag_stats"] = {
                    "total_chunks": total_chunks,
                    "retrieved_chunks": retrieved_chunks,
                }

            except Exception as error:
                st.session_state["contract_explanation"] = None
                st.session_state["contract_file_hash"] = None
                st.session_state["contract_sources"] = None
                st.session_state["contract_citations_valid"] = None
                st.session_state["contract_rag_stats"] = None

                st.error(t["contract_error"])
                print(
                    "Contract analysis error:",
                    repr(error),
                )

            finally:
                try:
                    os.remove(temp_path)
                except OSError:
                    pass

    cached_result_matches_file = (
        current_contract_hash is not None
        and st.session_state.get("contract_file_hash")
        == current_contract_hash
        and st.session_state.get("contract_explanation")
    )

    if cached_result_matches_file:
        render_html_card(f"""
            <div class="card">
                <div class="field-value" style="white-space: pre-wrap;">
                    {st.session_state['contract_explanation']}
                </div>
            </div>
        """)

        if not st.session_state.get(
            "contract_citations_valid",
            False,
        ):
            st.warning(t["citation_warning"])

        contract_sources = st.session_state.get(
            "contract_sources",
            [],
        )

        if contract_sources:
            with st.expander(t["sources_label"]):
                for source in contract_sources:
                    st.markdown(
                        f"**{t['source_label']} {source['id']} "
                        f"— {t['page_label']} {source['page']}**"
                    )
                    st.write(source["text"])
                    st.divider()

Overwriting /kaggle/working/app.py


## 6. Configure ngrok

Store `NGROK_AUTH_TOKEN` in Kaggle Secrets, then run this cell.


In [9]:
import os
import subprocess
import time

from pyngrok import conf, ngrok

try:
    from kaggle_secrets import UserSecretsClient
    NGROK_TOKEN = UserSecretsClient().get_secret(
        "NGROK_TOKEN"
    )
except Exception:
    NGROK_AUTH_TOKEN = os.getenv(
        "NGROK_TOKEN",
        "",
    )

if not NGROK_TOKEN:
    raise ValueError(
        "Add NGROK_TOKEN to Kaggle Secrets first."
    )

conf.get_default().auth_token = NGROK_TOKEN
print("ngrok token loaded.")


ngrok token loaded.


## 7. Start Streamlit


In [10]:
# Stop any old Streamlit process.
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
time.sleep(2)

# Start Streamlit in the background.
streamlit_proc = subprocess.Popen([
    "streamlit", "run", "/kaggle/working/app.py",
    "--server.port", "8501",
    "--server.headless", "true",
])
time.sleep(6)


2026-07-27 12:37:13.162 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://35.229.54.199:8501



## 8. Open the Public ngrok Link


In [11]:
# Open a public ngrok tunnel.
public_url = ngrok.connect(8501)
print("Streamlit app URL:", public_url)


Streamlit app URL: NgrokTunnel: "https://safari-junkyard-blighted.ngrok-free.dev" -> "http://localhost:8501"

INFO  ENV: Auto setting PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


/kaggle/working/app.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


from_quantized: adapter: None


Fetching 10 files: 100%|██████████| 10/10 [00:28<00:00,  2.89s/it]


INFO  Loader: Auto dtype (native bfloat16): `torch.bfloat16`                   
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: model_name_or_path.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: model_file_base_name.
INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]
INFO   Kernel: selected: `TritonV2QuantLinear`                                 
INFO  Kernel: candidates -> `[TritonV2QuantLinear]`                            
INFO  Kernel: selected -> `TritonV2QuantLinear`.                               
INFO  Format: Converting `checkpoint_format` from `gptq` to internal `gptq_v2`.
INFO  Format: Converting GPTQ v1 to v2                                         
INFO  Format: Conversion complete: 0.030699729919433594s                       
INFO   Kernel: selected: `TritonV2QuantLinear`                                 
INFO  Optimize: `TritonV2QuantLinear` compilation 

/kaggle/working/app.py:423: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  intent_chain = LLMChain(llm=llm, prompt=intent_prompt)
